In [29]:
from dual_spls.simulate import simulate
import numpy as np

In [30]:
## Configuration:
# Simulation configuration:
coefs = [0, 1, 0, 1, 1, 0, 0, 1, 1] 
n = 300
p = [1000]
nondes = [30]
sigmaondes = [0.30]
sigma_y = 0.05

# Split configuration:
split_mode = "random"

# Dual-sPLS configuration
norm = "pseudo-lasso" # options: "pseudo-lasso", ...

In [31]:
# generate synthetic data
simulation_results = simulate(n=n, p=p, nondes=nondes , sigmaondes=sigmaondes, sigma_y=sigma_y, coefs=coefs)
X, y = simulation_results["X"], simulation_results["y"]

In [32]:
# split data
if split_mode == "random":
    # split data randomly
    from sklearn.model_selection import train_test_split

    print("Splitting data using a classic random split.")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Splitting data using a classic random split.


In [33]:
# standardize data (after splitting to avoid data leakage)
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)       

y_scaler = StandardScaler()
y_train_scaled= y_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1)).flatten()


In [34]:
# apply Dual-sPLS
from dual_spls.lasso import dual_spls_lasso

results = {}
if norm == "pseudo-lasso":
    print("Applying Dual-sPLS with the \"pseudo-lasso\" norm.")

    for n_comp in range(1, 10):
        result = dual_spls_lasso(X_train_scaled, y_train_scaled, n_components=n_comp, ppnu=0.99)
        beta = result['Bhat'][:, -1]
        intercept = result['intercept'][-1]

        y_pred_scaled = (X_test_scaled @ beta + intercept).flatten()

        rmse = np.sqrt(np.mean((y_test_scaled - y_pred_scaled)**2))
        results[f"rmse_{n_comp}_comp"] = rmse

results

Applying Dual-sPLS with the "pseudo-lasso" norm.


{'rmse_1_comp': 0.18261500628620958,
 'rmse_2_comp': 0.1629064087253571,
 'rmse_3_comp': 0.02084291967187386,
 'rmse_4_comp': 0.0074042871602648585,
 'rmse_5_comp': 0.00039416836472361445,
 'rmse_6_comp': 0.00018066146064074453,
 'rmse_7_comp': 8.68366936588835e-05,
 'rmse_8_comp': 8.327352740695744e-05,
 'rmse_9_comp': 8.491908339151259e-05}

In [35]:
# apply PLS
from sklearn.cross_decomposition import PLSRegression

results_PLS = {}
for n_comp in range(1, 10):
        PLS = PLSRegression(n_components=n_comp, scale=False, copy=True)
        PLS.fit(X_train_scaled, y_train_scaled)

        y_pred_scaled = PLS.predict(X_test_scaled).flatten()

        rmse = np.sqrt(np.mean((y_test_scaled - y_pred_scaled)**2))
        results_PLS[f"rmse_{n_comp}_comp"] = rmse

results_PLS

{'rmse_1_comp': 0.06067741763855091,
 'rmse_2_comp': 0.018327041343306417,
 'rmse_3_comp': 0.013440004125246867,
 'rmse_4_comp': 0.0007437374941977493,
 'rmse_5_comp': 0.00016121480685943548,
 'rmse_6_comp': 9.255489693156278e-05,
 'rmse_7_comp': 8.394407617655474e-05,
 'rmse_8_comp': 8.330261615770453e-05,
 'rmse_9_comp': 8.507647343115016e-05}